# Shake It Up! — Test Suite

In [ ]:
import queue
import random
import statistics
import time

import serial
from serial.tools import list_ports

from serial_manager import SerialManager

print("Available serial ports:")
for p in list_ports.comports():
    print(f"  {p.device}  |  {p.description}")

In [ ]:
# Edit to match the firmware build:
#   MODE_DEBUG -> /dev/cu.HC-06 ...        @ 38400
#   MODE_USB   -> /dev/cu.usbmodemSDA...   @ 115200
#   MODE_BT    -> /dev/cu.HC-06 ...        @ 115200
SERIAL_PORT = "/dev/cu.usbmodemSDA6C1C1E501"
BAUD_RATE   = 115200

In [ ]:
ser_mgr = SerialManager(port=SERIAL_PORT, baud=BAUD_RATE)
assert ser_mgr.connect(), "could not open port"
print(f"Connected. Initial offset: {ser_mgr.sync.offset_ms:.1f} ms")


def drain_event_q():
    while not ser_mgr.dispatcher.event_q.empty():
        ser_mgr.dispatcher.event_q.get_nowait()


def drain_ack_q():
    while not ser_mgr.dispatcher.ack_q.empty():
        ser_mgr.dispatcher.ack_q.get_nowait()

In [ ]:
# Test 1 -- Round-trip latency.
# 30 SYN/ACK pings; report mean / median / max / std of the round-trip time.
# Expect: ~2-5 ms over USB-OpenSDA, ~20-60 ms over HC-06 Bluetooth.

N_PINGS = 30
rtts = []
for _ in range(N_PINGS):
    drain_ack_q()
    t0 = time.time()
    ser_mgr.dispatcher.write(b"SYN\n")
    try:
        ser_mgr.dispatcher.ack_q.get(timeout=1.0)
    except queue.Empty:
        continue
    rtts.append((time.time() - t0) * 1000)
    time.sleep(0.05)

print(f"  Samples:  {len(rtts)}")
print(f"  Mean:     {statistics.mean(rtts):6.2f} ms")
print(f"  Median:   {statistics.median(rtts):6.2f} ms")
print(f"  Max:      {max(rtts):6.2f} ms")
print(f"  Std:      {statistics.stdev(rtts):6.2f} ms")

In [ ]:
# Test 2 -- Clock-sync drift over 60 s.
# Watches the EMA-smoothed offset while the periodic-sync thread keeps pulling it back toward truth (every 5 s). 
# Reports total drift + ppm.

DURATION = 60
samples = []
t_end = time.time() + DURATION
while time.time() < t_end:
    off = ser_mgr.sync.offset_ms
    if off is not None:
        samples.append((time.time() * 1000, off))
    time.sleep(0.5)

if len(samples) >= 2:
    elapsed_s = (samples[-1][0] - samples[0][0]) / 1000
    drift_ms = samples[-1][1] - samples[0][1]
    drift_ppm = abs(drift_ms / elapsed_s) * 1000 if elapsed_s else 0
    print(f"  Samples:    {len(samples)}")
    print(f"  Elapsed:    {elapsed_s:.1f} s")
    print(f"  Net drift:  {drift_ms:+7.2f} ms (smoothed)")
    print(f"  Drift rate: {drift_ppm:7.0f} ppm")

In [ ]:
# Test 3 -- Idle false-trigger rate.
# Place the board flat and still; count SWING events over 60 s.
# Expect: produce 0 or very few spurious SWING events, and absolutely no H:/M:/BUSY: grades.

DURATION = 60
print(f"  Place the board flat and STILL. Listening for {DURATION} s...")

drain_event_q()
swing_count = 0
t_end = time.time() + DURATION
while time.time() < t_end:
    try:
        ev = ser_mgr.dispatcher.event_q.get(timeout=0.5)
    except queue.Empty:
        continue
    if ev["type"] == "swing":
        swing_count += 1
        print(f"  spurious SWING:{ev['dir']} @ board t={ev['t_ms']}")

print(f"\n  False triggers in {DURATION} s: {swing_count}")

In [ ]:
# Test 4 -- Gesture-recognition accuracy.
# Wave each direction N times in a randomized order, then reports per-direction accuracy.

DIR_NAMES = {"U": "Up", "D": "Down", "L": "Left", "R": "Right"}
N_PER_DIRECTION = 5

drain_event_q()
sequence = list("UDLR") * N_PER_DIRECTION
random.shuffle(sequence)

correct = {"U": 0, "D": 0, "L": 0, "R": 0}
detected_count = {"U": 0, "D": 0, "L": 0, "R": 0}

for i, prompt in enumerate(sequence, 1):
    print(f"  [{i:>2}/{len(sequence)}] Wave {DIR_NAMES[prompt]:5s} ", end="")
    for c in (3, 2, 1):
        time.sleep(1)
        print(f"{c} ", end="", flush=True)
    print(" SWING")

    drain_event_q()
    detected = None
    end = time.time() + 1.5
    while time.time() < end:
        try:
            ev = ser_mgr.dispatcher.event_q.get(timeout=0.1)
        except queue.Empty:
            continue
        if ev["type"] == "swing":
            detected = ev["dir"]
            break

    if detected is None:
        print(f"           (no swing detected)")
    else:
        detected_count[detected] += 1
        if detected == prompt:
            correct[prompt] += 1
            print(f"           OK detected {DIR_NAMES[detected]}")
        else:
            print(f"           XX detected {DIR_NAMES[detected]} (expected {DIR_NAMES[prompt]})")

total = sum(correct.values())
print()
print(f"  Overall accuracy: {total}/{len(sequence)} = {100*total/len(sequence):.0f}%")
for d in "UDLR":
    print(f"    {DIR_NAMES[d]:5s}: {correct[d]}/{N_PER_DIRECTION}")

In [ ]:
# Test 5 -- Timing-judgment consistency.
# Schedule one beat 3 s out, swing on the countdown.
# Verify the reported grade matches the measured |dt|.

PERFECT_MS, GOOD_MS, MISS_MS = 500, 1000, 1500   # mirror firmware

drain_event_q()
LOOKAHEAD = 3.0
direction = random.choice("UDLR")
host_target_ms = time.time() * 1000 + LOOKAHEAD * 1000
target_board_ms = ser_mgr.sync.host_to_board_ms(host_target_ms)
ser_mgr.send_beat(idx=42, direction=direction, host_target_ms=host_target_ms)
print(f"  Beat scheduled: {direction} (board t={target_board_ms} ms)")
print(f"  Get ready -- 3 ", end="")
for c in (3, 2, 1):
    time.sleep(1)
    print(f"{c} ", end="", flush=True)
print(" SWING")

end = time.time() + LOOKAHEAD + 2.5
result = None
while time.time() < end:
    try:
        ev = ser_mgr.dispatcher.event_q.get(timeout=0.1)
    except queue.Empty:
        continue
    if ev.get("idx") == 42 and ev["type"] in ("hit", "miss"):
        result = ev
        break

if result is None:
    print("\n  no result received")
elif result["type"] == "miss":
    print("\n  auto-MISS (no swing detected)")
else:
    dt = result["actual_ms"] - target_board_ms
    abs_dt = abs(dt)
    grade = result["grade"]
    expected = "P" if abs_dt <= PERFECT_MS else "G" if abs_dt <= GOOD_MS else "M"
    ok = (grade == expected)
    name = {"P": "PERFECT", "G": "GOOD", "M": "MISS"}
    print(f"  Reported:   {name[grade]}  (board returned '{grade}')")
    print(f"  Measured:   |dt| = {abs_dt} ms  (dt = {dt:+d})")
    print(f"  Expected:   {name[expected]}  given P<={PERFECT_MS}, G<={GOOD_MS}")
    print(f"  Consistent: {'PASS' if ok else 'FAIL'}")

In [ ]:
# Test 6 -- Auto-miss timeout.
# Schedule a beat, do NOT swing.
# Verify M:idx arrives at perfect_t + MISS_WINDOW.

LOOKAHEAD = 2.0
MISS_WINDOW = 1500
drain_event_q()
host_target_ms = time.time() * 1000 + LOOKAHEAD * 1000
ser_mgr.send_beat(idx=99, direction="U", host_target_ms=host_target_ms)
print("  Beat scheduled. DO NOT SWING.")

t_send = time.time()
end = t_send + LOOKAHEAD + 2.5
elapsed_ms = None
while time.time() < end:
    try:
        ev = ser_mgr.dispatcher.event_q.get(timeout=0.1)
    except queue.Empty:
        continue
    if ev.get("idx") == 99 and ev["type"] == "miss":
        elapsed_ms = (time.time() - t_send) * 1000
        break

expected_ms = LOOKAHEAD * 1000 + MISS_WINDOW
if elapsed_ms is None:
    print("  XX no auto-MISS received")
else:
    err = elapsed_ms - expected_ms
    print(f"  M:99 after {elapsed_ms:.0f} ms (expected ~{expected_ms:.0f}, error {err:+.0f} ms)")

In [ ]:
# Test 7 -- Queue-full BUSY.
# Send 17 beats with a small inter-beat gap so the board's 16-slot queue overflows on the 17th. 
# Expect: BUSY for #216 (the 17th sent), and auto-MISS for #200..#215 (the 16 that fit).

drain_event_q()
LOOKAHEAD = 2.0
INTER_BEAT_MS = 10
base_host = time.time() * 1000 + LOOKAHEAD * 1000

for i in range(17):
    ser_mgr.send_beat(idx=200 + i, direction="U",
                      host_target_ms=base_host + i * 50)
    time.sleep(INTER_BEAT_MS / 1000.0)

end = time.time() + LOOKAHEAD + 2.5
busies, misses = [], []
while time.time() < end:
    try:
        ev = ser_mgr.dispatcher.event_q.get(timeout=0.1)
    except queue.Empty:
        continue
    if 200 <= ev.get("idx", -1) < 220:
        if ev["type"] == "busy":
            busies.append(ev["idx"])
        elif ev["type"] == "miss":
            misses.append(ev["idx"])

expected_busies = [216]
expected_misses = list(range(200, 216))
ok = (sorted(busies) == expected_busies and sorted(misses) == expected_misses)
print(f"  BUSY received: {sorted(busies)}  (expected {expected_busies})")
print(f"  MISS received: {sorted(misses)}")
print(f"  {'PASS' if ok else 'FAIL'}")

In [ ]:
# Test 8 -- Stray-swing ignore.
# Wave the board for 5 s with an empty queue. 
# Expect SWING events but NO H:/M:/BUSY

drain_event_q()
DURATION = 5.0
print(f"  Wave the board around for {DURATION:.0f} s ...")

end = time.time() + DURATION
swings = bad = 0
while time.time() < end:
    try:
        ev = ser_mgr.dispatcher.event_q.get(timeout=0.1)
    except queue.Empty:
        continue
    if ev["type"] == "swing":
        swings += 1
    elif ev["type"] in ("hit", "miss", "busy"):
        bad += 1
        print(f"  XX unexpected {ev['type'].upper()}:{ev.get('idx')}")

print(f"  SWINGs detected: {swings}")
print(f"  Stray H/M/BUSY:  {bad}")
print(f"  {'PASS' if bad == 0 else 'FAIL'}")

In [ ]:
ser_mgr.disconnect()
print("Disconnected.")